In [12]:
from huggingface_hub import hf_hub_download
import pandas as pd
import json
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path
import re

## Parser Output Unification

Load all local parser output JSON files and build a unified schema across all 7 agents.

In [13]:
AGENTS = ["ag2", "appworld", "chatdev", "hyperagent", "magenticone", "metagpt", "openmanus"]

def load_parser_output(agent):
    path = Path(f"parsers/{agent}_parser/{agent}_output_mad.json")
    with open(path, encoding='utf-8') as f:
        return json.load(f)

raw = {agent: load_parser_output(agent) for agent in AGENTS}
print("Loaded traces per agent:")
for agent, traces in raw.items():
    print(f"  {agent:15s}: {len(traces)} traces")

Loaded traces per agent:
  ag2            : 597 traces
  appworld       : 30 traces
  chatdev        : 130 traces
  hyperagent     : 28 traces
  magenticone    : 195 traces
  metagpt        : 230 traces
  openmanus      : 30 traces


### Structural Overview

Print trace metadata keys, step kind values, and step metadata keys per agent.

In [14]:
for agent, traces in raw.items():
    meta_keys = set(traces[0]["metadata"].keys())
    step_meta_keys, step_kinds = set(), set()
    for t in traces:
        for step in t["steps"]:
            step_meta_keys.update(step.get("metadata", {}).keys())
            step_kinds.add(step.get("kind"))
    print("=" * 60)
    print(f"Agent: {agent}")
    print(f"  trace.metadata keys : {sorted(meta_keys)}")
    print(f"  step.kind values    : {sorted(step_kinds)}")
    print(f"  step.metadata keys  : {sorted(step_meta_keys)}")

Agent: ag2
  trace.metadata keys : ['agent_participation', 'answer', 'benchmark_name', 'final_answer', 'llm_name', 'mad_trace_key', 'mas_name', 'mast_annotation', 'n_agent_switches', 'n_turns', 'perturbation_type', 'success', 'task', 'trace_id']
  step.kind values    : ['message', 'system', 'tool_call', 'tool_result']
  step.metadata keys  : ['is_final_answer', 'step_index']
Agent: appworld
  trace.metadata keys : ['agent_participation', 'benchmark_name', 'failures', 'final_answer', 'final_status', 'llm_name', 'mad_trace_key', 'mas_name', 'mast_annotation', 'n_agent_switches', 'n_turns', 'passes', 'success', 'task', 'trace_id']
  step.kind values    : ['message', 'system', 'tool_call', 'tool_result']
  step.metadata keys  : ['is_final_answer', 'step_index']
Agent: chatdev
  trace.metadata keys : ['agent_participation', 'benchmark_name', 'final_answer', 'llm_name', 'mad_trace_key', 'mas_name', 'mast_annotation', 'n_agent_switches', 'n_turns', 'phases', 'task', 'trace_id']
  step.kind va

### Field Coverage Analysis

For each metadata and step-metadata field, count how many agents have it.

In [15]:
# trace-level metadata
meta_field_agents = defaultdict(list)
for agent, traces in raw.items():
    for key in traces[0]["metadata"].keys():
        meta_field_agents[key].append(agent)

meta_coverage = pd.DataFrame([
    {"field": k, "n_agents": len(v), "agents": ", ".join(sorted(v))}
    for k, v in meta_field_agents.items()
]).sort_values(["n_agents", "field"], ascending=[False, True]).reset_index(drop=True)

print("Trace-level metadata field coverage:")
print(meta_coverage.to_string(index=False))

Trace-level metadata field coverage:
              field  n_agents                                                              agents
     benchmark_name         7 ag2, appworld, chatdev, hyperagent, magenticone, metagpt, openmanus
           llm_name         7 ag2, appworld, chatdev, hyperagent, magenticone, metagpt, openmanus
      mad_trace_key         7 ag2, appworld, chatdev, hyperagent, magenticone, metagpt, openmanus
           mas_name         7 ag2, appworld, chatdev, hyperagent, magenticone, metagpt, openmanus
    mast_annotation         7 ag2, appworld, chatdev, hyperagent, magenticone, metagpt, openmanus
            n_turns         7 ag2, appworld, chatdev, hyperagent, magenticone, metagpt, openmanus
               task         7 ag2, appworld, chatdev, hyperagent, magenticone, metagpt, openmanus
           trace_id         7 ag2, appworld, chatdev, hyperagent, magenticone, metagpt, openmanus
agent_participation         6            ag2, appworld, chatdev, hyperagent, mage

In [16]:
# step-level metadata
step_meta_field_agents = defaultdict(list)
for agent, traces in raw.items():
    keys_seen = set()
    for t in traces:
        for step in t["steps"]:
            keys_seen.update(step.get("metadata", {}).keys())
    for key in keys_seen:
        step_meta_field_agents[key].append(agent)

step_meta_coverage = pd.DataFrame([
    {"field": k, "n_agents": len(v), "agents": ", ".join(sorted(v))}
    for k, v in step_meta_field_agents.items()
]).sort_values(["n_agents", "field"], ascending=[False, True]).reset_index(drop=True)

print("Step-level metadata field coverage:")
print(step_meta_coverage.to_string(index=False))

Step-level metadata field coverage:
           field  n_agents                                                              agents
      step_index         7 ag2, appworld, chatdev, hyperagent, magenticone, metagpt, openmanus
 is_final_answer         4                              ag2, appworld, hyperagent, magenticone
      agent_pair         1                                                             chatdev
      event_type         1                                                           openmanus
        is_stuck         1                                                           openmanus
n_tools_selected         1                                                           openmanus
      phase_name         1                                                             chatdev
          step_n         1                                                           openmanus
       tool_args         1                                                           openmanus
       tool_na

### Unified Schema

**Common trace metadata fields** (present in all or most agents):
, , , , , , , , , , , 

Agent-specific fields are collected into an  dict.

**Common step fields**:
, , , , , , 

Agent-specific step-metadata fields go into .

In [17]:
COMMON_TRACE_FIELDS = [
    "trace_id", "mad_trace_key", "mas_name", "llm_name", "benchmark_name",
    "task", "final_answer", "success", "n_turns", "n_agent_switches",
    "agent_participation", "mast_annotation",
]
COMMON_STEP_META_FIELDS = ["step_index", "is_final_answer"]


def normalize_trace(trace):
    meta = trace["metadata"]
    unified_meta = {field: meta.get(field) for field in COMMON_TRACE_FIELDS}
    unified_meta["trace_id"] = str(unified_meta["trace_id"])
    unified_meta["extra"] = {k: v for k, v in meta.items() if k not in COMMON_TRACE_FIELDS}

    unified_steps = []
    for i, step in enumerate(trace["steps"]):
        sm = step.get("metadata", {})
        unified_steps.append({
            "step_index"     : sm.get("step_index", i),
            "agent"          : step.get("agent"),
            "kind"           : step.get("kind"),
            "content"        : step.get("content"),
            "is_final_answer": bool(sm.get("is_final_answer", False)),
            "extra"          : {k: v for k, v in sm.items() if k not in COMMON_STEP_META_FIELDS},
        })
    return {"metadata": unified_meta, "steps": unified_steps}


unified = {agent: [normalize_trace(t) for t in traces] for agent, traces in raw.items()}

print("Normalized traces per agent:")
for agent, traces in unified.items():
    print(f"  {agent:15s}: {len(traces)} traces")

# spot-check
sample = unified["ag2"][0]
print("Unified trace metadata keys:", list(sample["metadata"].keys()))
print("Unified step keys          :", list(sample["steps"][0].keys()))

Normalized traces per agent:
  ag2            : 597 traces
  appworld       : 30 traces
  chatdev        : 130 traces
  hyperagent     : 28 traces
  magenticone    : 195 traces
  metagpt        : 230 traces
  openmanus      : 30 traces
Unified trace metadata keys: ['trace_id', 'mad_trace_key', 'mas_name', 'llm_name', 'benchmark_name', 'task', 'final_answer', 'success', 'n_turns', 'n_agent_switches', 'agent_participation', 'mast_annotation', 'extra']
Unified step keys          : ['step_index', 'agent', 'kind', 'content', 'is_final_answer', 'extra']


### Flat DataFrames for Analysis

In [18]:
trace_rows, step_rows = [], []

for agent, traces in unified.items():
    for trace in traces:
        m = trace["metadata"]
        trace_rows.append({
            "agent_name"      : agent,
            "trace_id"        : m["trace_id"],
            "mad_trace_key"   : m["mad_trace_key"],
            "mas_name"        : m["mas_name"],
            "llm_name"        : m["llm_name"],
            "benchmark_name"  : m["benchmark_name"],
            "task"            : m["task"],
            "final_answer"    : m["final_answer"],
            "success"         : m["success"],
            "n_turns"         : m["n_turns"],
            "n_agent_switches": m["n_agent_switches"],
            "n_steps"         : len(trace["steps"]),
        })
        for step in trace["steps"]:
            step_rows.append({
                "agent_name"     : agent,
                "trace_id"       : m["trace_id"],
                "step_index"     : step["step_index"],
                "agent"          : step["agent"],
                "kind"           : step["kind"],
                "content"        : step["content"],
                "is_final_answer": step["is_final_answer"],
            })

df_traces = pd.DataFrame(trace_rows)
df_steps  = pd.DataFrame(step_rows)

print(f"df_traces : {df_traces.shape[0]:,} rows x {df_traces.shape[1]} cols")
print(f"df_steps  : {df_steps.shape[0]:,} rows x {df_steps.shape[1]} cols")
df_traces.head()

df_traces : 1,240 rows x 12 cols
df_steps  : 26,332 rows x 7 cols


,agent_name,trace_id,mad_trace_key,mas_name,llm_name,benchmark_name,task,final_answer,success,n_turns,n_agent_switches,n_steps
0,ag2,0,AG2_GSM_Plus_GPT4o,AG2,GPT-4o,GSM,Joey has 214 points before his turn in Scrabbl...,9,False,4,3.0,4
1,ag2,1,AG2_GSM_Plus_GPT4o,AG2,GPT-4o,GSM,Jane planted a beanstalk in her backyard. Afte...,2h + 4,True,4,3.0,4
2,ag2,2,AG2_GSM_Plus_GPT4o,AG2,GPT-4o,GSM,Monica is wrapping Christmas gifts. She has 6 ...,"\text{""Data insufficient to solve the problem""}",True,10,9.0,10
3,ag2,3,AG2_GSM_Plus_GPT4o,AG2,GPT-4o,GSM,"Of the 20 available cars for rent, 12 are auto...",26.67\%,True,6,5.0,6
4,ag2,4,AG2_GSM_Plus_GPT4o,AG2,GPT-4o,GSM,Courtney attended a concert and reported that ...,8,False,4,3.0,4


## Failure Mode Detection

Given a MAST failure mode key (e.g. `"1.1"`), return every trace that has that failure mode annotated as `1`.
Because `trace_id` is only unique within an agent, results include `(agent_name, trace_id)` pairs.

In [19]:
def detect_failure_mode(failure_mode: str, data: dict = unified) -> list[dict]:
    """
    Return all traces where `mast_annotation[failure_mode] == 1`.

    Parameters
    ----------
    failure_mode : str
        MAST annotation key, e.g. "1.1", "2.3", "3.2".
    data : dict
        Mapping of agent_name -> list of unified traces (defaults to `unified`).

    Returns
    -------
    List of dicts with keys: agent_name, trace_id, task, success, mast_annotation.
    """
    results = []
    for agent_name, traces in data.items():
        for trace in traces:
            annotation = trace["metadata"].get("mast_annotation") or {}
            if annotation.get(failure_mode) == 1:
                results.append({
                    "agent_name"     : agent_name,
                    "trace_id"       : trace["metadata"]["trace_id"],
                    "task"           : trace["metadata"]["task"],
                    "success"        : trace["metadata"]["success"],
                    "mast_annotation": annotation,
                })
    return results


# --- example usage ---
matches = detect_failure_mode("1.1")

In [20]:
# Summary: count of traces with each failure mode set, broken out by agent
all_keys = sorted(
    {k for traces in unified.values() for t in traces
     for k in (t["metadata"].get("mast_annotation") or {}).keys()}
)

rows = []
for fm in all_keys:
    row = {"failure_mode": fm}
    for agent in AGENTS:
        row[agent] = sum(
            1 for t in unified[agent]
            if (t["metadata"].get("mast_annotation") or {}).get(fm) == 1
        )
    row["total"] = sum(row[a] for a in AGENTS)
    rows.append(row)

df_fm = pd.DataFrame(rows).set_index("failure_mode")
print("Failure mode hit counts per agent:")
df_fm

Failure mode hit counts per agent:


,ag2,appworld,chatdev,hyperagent,magenticone,metagpt,openmanus,total
failure_mode,,,,,,,,
1.1,193,7,32,7,64,57,7,367
1.2,8,0,0,0,2,0,0,10
1.3,232,6,47,6,66,88,6,451
1.4,45,1,5,1,9,9,1,71
1.5,177,2,38,2,51,74,2,346
2.1,30,1,4,1,6,7,1,50
2.2,170,2,27,2,48,52,2,303
2.3,129,2,19,2,34,36,2,224
2.4,21,1,4,1,5,7,1,40


## Trace Export

Export matched traces to a human-readable markdown file. Open the output file in VS Code with **Markdown Preview** (`Cmd+Shift+V`) to read through traces comfortably.

In [21]:
import random

def export_traces_markdown(failure_mode: str, output_path: str = None, data: dict = None, max_traces: int = 200, seed: int = 42) -> str:
    """
    Export a random sample of traces with a given failure mode to a readable markdown file.

    Parameters
    ----------
    failure_mode : str   e.g. "1.1"
    output_path  : str   defaults to traces_failure_<failure_mode>.md
    data         : dict  agent_name -> unified traces (defaults to `unified`)
    max_traces   : int   cap on how many traces to write (randomly sampled, default 20)
    seed         : int   random seed for reproducibility
    """
    if data is None:
        data = unified
    if output_path is None:
        output_path = f"traces_failure_{failure_mode.replace('.', '_')}.md"

    all_matches = detect_failure_mode(failure_mode, data)
    rng = random.Random(seed)
    matches = rng.sample(all_matches, min(max_traces, len(all_matches)))

    trace_lookup = {
        (agent, trace["metadata"]["trace_id"]): trace
        for agent, traces in data.items()
        for trace in traces
    }

    KIND_LABEL = {
        "message"    : "Message",
        "tool_call"  : "Tool Call",
        "tool_result": "Tool Result",
        "system"     : "System",
    }

    def indent_block(text: str) -> str:
        """Indent every line with 4 spaces — avoids nested backtick fence issues."""
        return "\n".join("    " + line for line in text.splitlines()) if text else "    (empty)"

    lines = []
    lines.append(f"# Traces with failure mode `{failure_mode}`\n")
    lines.append(f"**Total matches:** {len(all_matches)} &nbsp;|&nbsp; **Showing:** {len(matches)} (randomly sampled, seed={seed})  \n")
    lines.append("---\n")

    for i, match in enumerate(matches, 1):
        agent_name = match["agent_name"]
        trace_id   = match["trace_id"]
        trace      = trace_lookup[(agent_name, trace_id)]
        m          = trace["metadata"]

        lines.append(f"## Trace {i} — `{agent_name}` / `{trace_id}`\n")

        lines.append("| Field | Value |")
        lines.append("|---|---|")
        lines.append(f"| **Agent system** | {m.get('mas_name', agent_name)} |")
        lines.append(f"| **LLM** | {m.get('llm_name', '—')} |")
        lines.append(f"| **Benchmark** | {m.get('benchmark_name', '—')} |")
        lines.append(f"| **Success** | {m.get('success')} |")
        lines.append(f"| **Turns** | {m.get('n_turns')} |")
        lines.append(f"| **Agent switches** | {m.get('n_agent_switches')} |")
        ann = m.get('mast_annotation') or {}
        active = [k for k, v in sorted(ann.items()) if v == 1]
        lines.append(f"| **Active failure modes** | {', '.join(active) if active else 'none'} |")
        lines.append("")

        lines.append("**Task**")
        lines.append(">" + (m.get('task') or '').replace('\n', '  \n>'))
        lines.append("")

        if m.get('final_answer'):
            lines.append("**Final answer**")
            lines.append(">" + str(m['final_answer']).replace('\n', '  \n>'))
            lines.append("")

        lines.append("### Steps\n")
        for step in trace["steps"]:
            label   = KIND_LABEL.get(step["kind"], step["kind"])
            content = (step["content"] or "").strip()
            lines.append(f"#### Step {step['step_index']} &nbsp;·&nbsp; {label} &nbsp;·&nbsp; `{step['agent']}`\n")
            lines.append(indent_block(content))
            lines.append("")

        lines.append("---\n")

    with open(output_path, "w", encoding='utf-8') as f:
        f.write("\n".join(lines))

    print(f"Wrote {len(matches)}/{len(all_matches)} traces → {output_path}")
    return output_path


# --- example usage ---
export_traces_markdown("3.1")


Wrote 200/208 traces → traces_failure_3_1.md


'traces_failure_3_1.md'